# ADOFAI Android 3.2.1 — 비공개 커스텀 게임 APK *시험 빌드*

실제 Google Drive APK는 **versionName 3.2.1 / versionCode 300381**입니다. 파일명에 `3.3.1`이 들어 있어도 그 숫자를 신뢰하지 않습니다.
위에서 **런타임 → 모두 실행**을 누르고 본인 Google Drive 접근을 승인하세요. **PC에는 어떤 것도 설치하지 않습니다.** 드라이브 원본은 수정하지 않고 결과를 별도 파일로 저장합니다.

이 프로젝트의 에디터 네이티브 브리지는 원래 3.3.1 대상으로 개발되었습니다. **3.2.1에서 게임/커스텀 맵 실행 호환성은 미검증**입니다. 이 노트북은 시험용 APK만 만들며, 설치 전에 게임 데이터 백업과 분리된 테스트 환경이 필요합니다.


In [ ]:
from google.colab import drive
from pathlib import Path
import subprocess, shutil, zipfile, json, hashlib, secrets, os, importlib.util, urllib.request
drive.mount('/content/drive')
root=Path('/content/drive/MyDrive')
original=root/'A_Dance_Of_Fire_And_Ice_3.3.1_1787072462_latestmodapks.com.apk'
folder=root/'ADOFAI 3.3.1 - Cloud Custom Build'
payload_zip=folder/'Editor_Payload.zip'
work=Path('/content/adofai_custom_321')
work.mkdir(parents=True,exist_ok=True)
assert original.is_file(), '원본 Android APK가 Google Drive 최상위에 없습니다.'
assert payload_zip.is_file(), 'Editor_Payload.zip 파일이 빌드 폴더에 없습니다.'
def run(*args, cwd=None):
    subprocess.run([str(a) for a in args],cwd=str(cwd) if cwd else None,check=True)
print('Google Drive에서 원본 읽기 준비:',original.stat().st_size,'bytes')
print('PC에 설치/다운로드하지 않습니다. 모든 작업은 Colab 임시 디스크에서 실행합니다.')


In [ ]:
local=work/'source.apk'
if local.stat().st_size != original.stat().st_size if local.exists() else True:
    with original.open('rb') as src,local.open('wb') as dst:
        shutil.copyfileobj(src,dst,4*1024*1024)
assert local.stat().st_size==original.stat().st_size
repo=work/'repo'
if not repo.exists():
    run('git','clone','--depth','1','--branch','feat/331-drive-apk-source','https://github.com/hoonex/adofai.git',repo)
spec=importlib.util.spec_from_file_location('adofai_audit',repo/'tools/apk_remote_audit.py')
audit=importlib.util.module_from_spec(spec)
spec.loader.exec_module(audit)
with zipfile.ZipFile(local) as z:
    files=set(z.namelist())
    m=audit.manifest_info(z.read('AndroidManifest.xml'))
    assert m.get('package')=='com.fizzd.connectedworlds',m
    assert str(m.get('versionName'))=='3.2.1',m
    assert str(m.get('versionCode'))=='300381',m
    for n in ('classes.dex','lib/arm64-v8a/libil2cpp.so','lib/arm64-v8a/libunity.so'):
        assert n in files, '필수 Android 런타임 파일 누락: '+n
    for n in ('classes2.dex','lib/arm64-v8a/libOctober.so'):
        assert n not in files, '이미 다른 모드가 삽입되어 있어 안전상 중단: '+n
print('실제 원본 3.2.1(300381) 버전 검사 완료:',m)

# Native 3.3.1 브리지가 참조하는 필수 이름들이 3.2.1 메타데이터에도 있는지 사전 검사.
# 이름 존재만으로 시그니처/ABI 호환성이 검증되지는 않습니다.
with zipfile.ZipFile(local) as original_apk:
    metadata_path='assets/bin/Data/Managed/Metadata/global-metadata.dat'
    assert metadata_path in original_apk.namelist(), '원본 IL2CPP 메타데이터 누락: 이 버전은 자동 패치하지 않습니다.'
    metadata_bytes=original_apk.read(metadata_path)
    for symbol in (b'scrController', b'LoadCustomLevel', b'GCS', b'customLevelIndex', b'get_touchCount'):
        assert symbol in metadata_bytes, '3.2.1 메타데이터에서 필요한 IL2CPP 이름을 못 찾았습니다: '+symbol.decode()
print('정적 메타데이터 필수 이름 확인 완료. 실제 네이티브 ABI 동작 검증은 별개입니다.')
payload=work/'payload'
payload.mkdir(exist_ok=True)
with zipfile.ZipFile(payload_zip) as z:
    hashes=dict(line.split() for line in z.read('SHA256SUMS.txt').decode().splitlines())
    for name in ('classes2.dex','libOctober.so'):
        blob=z.read(name)
        assert hashlib.sha256(blob).hexdigest()==hashes[name]
        (payload/name).write_bytes(blob)
assert (payload/'classes2.dex').read_bytes()[:4]==b'dex\n'
assert (payload/'libOctober.so').read_bytes()[:4]==b'\x7fELF'
print('Editor DEX/ARM64 payload SHA256 검증 성공')


In [ ]:
JAVA_SOURCE="package dev.hoonex.adofai.gamepatcher;\nimport java.io.File;\nimport java.nio.file.Path;\nimport java.util.zip.ZipFile;\npublic final class CloudPatcherMain {\n  public static void main(String[] a) throws Exception {\n    if (a.length!=5) throw new IllegalArgumentException(\"source dex lib output work\");\n    File input=Path.of(a[0]).toFile(), dex=Path.of(a[1]).toFile(), lib=Path.of(a[2]).toFile(),\n         output=Path.of(a[3]).toFile(), work=Path.of(a[4]).toFile();\n    if(input.getCanonicalPath().equals(output.getCanonicalPath())) throw new IllegalArgumentException(\"cannot overwrite source\");\n    if(!work.isDirectory() && !work.mkdirs()) throw new IllegalStateException(\"create work directory failed\");\n    try(ZipFile zip=new ZipFile(input)) {\n      for(String n:new String[]{\"classes.dex\",\"AndroidManifest.xml\",\"lib/arm64-v8a/libil2cpp.so\",\"lib/arm64-v8a/libunity.so\"})\n        if(zip.getEntry(n)==null) throw new IllegalArgumentException(\"missing source \"+n);\n      for(String n:new String[]{\"classes2.dex\",\"lib/arm64-v8a/libOctober.so\"})\n        if(zip.getEntry(n)!=null) throw new IllegalArgumentException(\"refusing preexisting payload \"+n);\n    }\n    ApkMutator.mutateBase(input,output,dex,work);\n    ApkMutator.injectArm64IntoExisting(output,lib);\n    for(String n:new String[]{\"classes.dex\",\"classes2.dex\",\"AndroidManifest.xml\",\"lib/arm64-v8a/libOctober.so\"})\n      ApkMutator.assertEntry(output,n);\n    System.out.println(\"Patched APK ZIP prepared: \"+output);\n  }\n}"
GRADLE_SOURCE="plugins { id 'application' }\nrepositories { google(); mavenCentral() }\ndependencies {\n implementation 'org.smali:dexlib2:2.5.2'\n implementation 'com.google.guava:guava:31.1-jre'\n implementation 'com.android:zipflinger:8.7.3'\n implementation 'io.github.reandroid:ARSCLib:1.4.0'\n}\napplication { mainClass = 'dev.hoonex.adofai.gamepatcher.CloudPatcherMain' }\njava { toolchain { languageVersion = JavaLanguageVersion.of(17) } }"
# 필요한 Java/Gradle 도구는 Colab 서버에만 준비합니다.
gradle_zip=work/'gradle-8.9-bin.zip'
gradle=work/'gradle-8.9/bin/gradle'
if not gradle.exists():
    if not gradle_zip.exists():
        urllib.request.urlretrieve('https://services.gradle.org/distributions/gradle-8.9-bin.zip',gradle_zip)
    with zipfile.ZipFile(gradle_zip) as z:
        z.extractall(work)
build=work/'cloud-cli'
java_dir=build/'src/main/java/dev/hoonex/adofai/gamepatcher'
java_dir.mkdir(parents=True,exist_ok=True)
upstream=repo/'android/game-patcher/app/src/main/java/dev/hoonex/adofai/gamepatcher'
for name in ('ApkMutator.java','DexBootstrapPatcher.java','V240DexBootstrapPatcher.java','ManifestStoragePatcher.java'):
    shutil.copy2(upstream/name,java_dir/name)
(java_dir/'CloudPatcherMain.java').write_text(JAVA_SOURCE,encoding='utf8')
(java_dir/'InstalledGame.java').write_text('package dev.hoonex.adofai.gamepatcher; final class InstalledGame { static final String PACKAGE_NAME="com.fizzd.connectedworlds"; }\n')
(build/'build.gradle').write_text(GRADLE_SOURCE)
(build/'settings.gradle').write_text("pluginManagement { repositories { gradlePluginPortal(); google(); mavenCentral() } }\nrootProject.name='adofai321-cloud-patcher'\n")
run(gradle,'--no-daemon','-p',build,'installDist')
patcher=build/'build/install/adofai321-cloud-patcher/bin/adofai321-cloud-patcher'
unsigned=work/'adofai-321-custom-unsigned.apk'
run(patcher,local,payload/'classes2.dex',payload/'libOctober.so',unsigned,work/'apk-work')
print('수정 APK 구조 생성:',unsigned.stat().st_size)


In [ ]:
# Colab 서버에만 Android 서명 도구를 설치합니다.
run('apt-get','update','-qq')
run('apt-get','install','-y','-qq','apksigner','zipalign','openjdk-17-jdk-headless')
run('zipalign','-c','-p','4',unsigned)
keystore=folder/'custom-signing.keystore'
password_file=folder/'custom-signing-passphrase.txt'
assert keystore.exists()==password_file.exists(), '기존 서명키 구성이 불완전합니다. 파일을 삭제하지 않고 중단합니다.'
if not keystore.exists():
    passphrase=secrets.token_urlsafe(30)
    password_file.write_text(passphrase,encoding='utf8')
    run('keytool','-genkeypair','-keystore',keystore,'-storepass',passphrase,
        '-keypass',passphrase,'-alias','adofai-cloud','-keyalg','RSA','-keysize','3072',
        '-validity','10000','-dname','CN=ADOFAI Private Build,O=Local,C=KR','-noprompt')
password=password_file.read_text().strip()
signed=work/'ADOFAI-3.2.1-Custom-UNVERIFIED.apk'
run('apksigner','sign','--ks',keystore,'--ks-key-alias','adofai-cloud','--ks-pass','pass:'+password,
    '--key-pass','pass:'+password,'--v4-signing-enabled','false','--out',signed,unsigned)
run('apksigner','verify','--verbose',signed)
def entry_sha(z,name):
    h=hashlib.sha256()
    with z.open(name) as f:
        for chunk in iter(lambda:f.read(4*1024*1024),b''):h.update(chunk)
    return h.hexdigest()
with zipfile.ZipFile(local) as old,zipfile.ZipFile(signed) as new:
    for name in ('lib/arm64-v8a/libil2cpp.so','lib/arm64-v8a/libunity.so','lib/arm64-v8a/libmain.so','assets/bin/Data/Managed/Metadata/global-metadata.dat'):
        if name in old.namelist():
            assert name in new.namelist() and entry_sha(old,name)==entry_sha(new,name), '원본 바이너리가 바뀜: '+name
    assert new.getinfo('classes2.dex').file_size>0
    assert new.getinfo('lib/arm64-v8a/libOctober.so').file_size>0
output=folder/'ADOFAI-3.2.1-Custom-UNVERIFIED.apk'
with signed.open('rb') as src,output.open('wb') as dst:
    shutil.copyfileobj(src,dst,4*1024*1024)
assert output.stat().st_size==signed.stat().st_size
h=hashlib.sha256()
with signed.open('rb') as f:
    for chunk in iter(lambda:f.read(4*1024*1024),b''):h.update(chunk)
(folder/'build-report.json').write_text(json.dumps({'source':original.name,'output':output.name,'size':output.stat().st_size,'sha256':h.hexdigest(),'sourceVersionName':'3.2.1','sourceVersionCode':300381,'gameplayVerified':False,'nativeBridgeCompatibility':'UNVERIFIED_331_ORIGIN_PAYLOAD'},indent=2))
print('Google Drive에 수정 게임 APK 저장:',output)
print('중요: 이 APK는 3.2.1 시험 빌드이고, 3.3.1용 네이티브 브리지 호환성 및 실제 플레이는 미검증입니다.')
